# E-Value Example

This notebook shows the first-class e-value lane: batch e-BH, online e-LOND, construction helpers, and Gaussian likelihood-ratio e-value simulation.

In [1]:
from online_fdr.e_values import EBH, e_bh

e_values = [1.0, 4.0, 80.0, 12.0, 0.5, 25.0]

functional_decisions = e_bh(e_values, alpha=0.1)

batch_method = EBH(alpha=0.1)
stateful_decisions = batch_method.test_batch(e_values)

print(functional_decisions)
print(stateful_decisions)
print(batch_method.current_threshold)

[False, False, True, False, False, False]
[False, False, True, False, False, False]
60.0


In [2]:
from online_fdr.e_values import ELond

stream_method = ELond(alpha=0.1)
stream = [1.0, 4.0, 80.0, 2.0, 500.0]

for idx, e_value in enumerate(stream, start=1):
    rejected = stream_method.test_one(e_value)
    print(
        idx,
        f"e={e_value:.1f}",
        f"level={stream_method.current_level:.6g}",
        f"threshold={stream_method.current_threshold:.3f}",
        rejected,
    )

1 e=1.0 level=0.00535168 threshold=186.857 False
2 e=4.0 level=0.00116382 threshold=859.239 False
3 e=80.0 level=0.00099125 threshold=1008.827 False
4 e=2.0 level=0.000824361 threshold=1213.061 False
5 e=500.0 level=0.000698887 threshold=1430.847 False


In [3]:
from online_fdr.e_values import e_to_p, make_power_calibrator, weighted_arithmetic_mean

p_values = [0.001, 0.2, 0.03, 0.8, 0.01]
calibrator = make_power_calibrator(exponent=0.5)
calibrated_e_values = [calibrator(p_value) for p_value in p_values]
conservative_p_values = [e_to_p(e_value) for e_value in calibrated_e_values]
merged = weighted_arithmetic_mean([2.0, 0.8, 5.0], weights=[1.0, 1.0, 2.0])

print(calibrated_e_values)
print(conservative_p_values)
print(merged)

[15.811388300841896, 1.118033988749895, 2.8867513459481287, 0.5590169943749475, 5.0]
[0.06324555320336758, 0.8944271909999159, 0.34641016151377546, 1.0, 0.2]
3.2


In [4]:
from online_fdr.e_values import ELond, GaussianEValueGenerator

generator = GaussianEValueGenerator(n=40, pi0=0.85, alt_mean=3.0, seed=7)
method = ELond(alpha=0.1)

true_discoveries = 0
false_discoveries = 0

for _ in range(40):
    e_value, is_alternative = generator.sample_one()
    rejected = method.test_one(e_value)
    true_discoveries += int(rejected and is_alternative)
    false_discoveries += int(rejected and not is_alternative)

discoveries = true_discoveries + false_discoveries
empirical_fdr = false_discoveries / max(discoveries, 1)

print(true_discoveries, false_discoveries, empirical_fdr)

2 0 0.0
